In [1]:
from os.path import commonprefix
import attr
import json
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
from argparse import ArgumentParser
from src.utils import load_df
import pandas as pd
from src.structs import LemmatizeRule, WordForm
import attr
import pandas as pd

from src.structs import LemmatizeRule, WordForm
from src.utils import load_df

In [4]:
data_path = Path("/mnt/data_storage/datasets/generative_lemmatization_datasets/csvs")

In [5]:
# в test берем dev сплит чтобы в замерах финального пайплайна на настоящем test сплите в других работах не было утечки
df_train = load_df(data_path / "train.csv", dedup=True)
df_test = load_df(data_path / "dev.csv", dedup=True)

df_train = df_train.drop(["subset", "filename"], axis=1)
df_test = df_test.drop(["subset", "filename"], axis=1)

In [6]:
df_train.info()

<class 'pandas.DataFrame'>
Index: 254267 entries, 0 to 2150052
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   form    254267 non-null  str  
 1   pos     254267 non-null  str  
 2   feats   252178 non-null  str  
 3   lemma   254267 non-null  str  
dtypes: str(4)
memory usage: 9.7 MB


In [7]:
df_test.info()

<class 'pandas.DataFrame'>
Index: 52959 entries, 0 to 255990
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   form    52959 non-null  str  
 1   pos     52959 non-null  str  
 2   feats   51794 non-null  str  
 3   lemma   52959 non-null  str  
dtypes: str(4)
memory usage: 2.0 MB


In [8]:
df_train["feats"] = df_train["feats"].fillna("No Feats")
df_test["feats"] = df_test["feats"].fillna("No Feats")

In [10]:
# у теста и трейна есть пересечения (высокочастотные слова и пунктуация). 
# поэтому test дополнитлеьно разделим на то что есть в train (known) и то чего нет (unknown)

cols = ['form', 'pos', 'feats', 'lemma']

key_train = df_train[cols].astype(str).agg("|".join, axis=1)
key_test = df_test[cols].astype(str).agg("|".join, axis=1)

mask = key_test.isin(key_train)

df_test_known = df_test[mask]
df_test_unkn = df_test[~mask]

In [14]:
print(f"Total test size: {df_test.shape[0]}; known size: {df_test_known.shape[0]}; unknown size: {df_test_unkn.shape[0]}")
assert df_test_known.shape[0] + df_test_unkn.shape[0] == df_test.shape[0]

Total test size: 52959; known size: 32558; unknown size: 20401


In [12]:
df_test_known.head()

,form,pos,feats,lemma
2,!,PUNCT,No Feats,!
5,Мы,PRON,Case:Nom Number:Plur Person:1 PronType:Prs,мы
6,Алиса,PROPN,Animacy:Anim Case:Nom Gender:Fem NameType:Giv ...,Алиса
7,и,CCONJ,No Feats,и
8,Коля,PROPN,Animacy:Anim Case:Nom Gender:Masc NameType:Giv...,Коля


In [13]:
df_test_unkn.head()

,form,pos,feats,lemma
0,🤘,SYM,No Feats,🤘
1,Хей,INTJ,No Feats,хей
3,Ставь,VERB,Aspect:Imp Mood:Imp Number:Sing Person:2 Trans...,ставить
4,❤️,SYM,No Feats,❤️
11,#безумных,ADJ,Case:Gen Degree:Pos Number:Plur,#безумный


In [ ]:
## таким образом в known test split попала частотная лексика, которая естественным образом появляется и в трейне (из-за св-в естественного языка и речи)
## а в unknown test split попадают менее частотные случаи, гапаксы в большом кол-ве и тд

In [ ]:
# from https://github.com/DanAnastasyev/GramEval2020/blob/master/solution/train/lemmatize_helper.py
def predict_lemmatize_rule(word: str, lemma: str):
    def _predict_lemmatize_rule(word: str, lemma: str, **kwargs):
        if len(word) == 0:
            return LemmatizeRule(append_suffix=lemma, **kwargs)

        common_prefix = commonprefix([word, lemma])
        if len(common_prefix) == 0:
            rule = _predict_lemmatize_rule(word[1:], lemma, **kwargs)
            return attr.evolve(rule, cut_prefix=rule.cut_prefix + 1)

        return LemmatizeRule(cut_suffix=len(word) - len(common_prefix),
                                append_suffix=lemma[len(common_prefix):], **kwargs)

    word, lemma = word.replace('ё', 'е'), lemma.replace('ё', 'е')
    return min([
        _predict_lemmatize_rule(word, lemma),
        _predict_lemmatize_rule(word.lower(), lemma, lower=True),
        _predict_lemmatize_rule(word.capitalize(), lemma, capitalize=True),
        _predict_lemmatize_rule(word.upper(), lemma, upper=True)
    ], key=lambda rule: rule.cut_prefix + rule.cut_suffix)



def _sort_rules(rules):
    return sorted(
        rules, key=lambda r: (
            r.cut_prefix, r.cut_suffix, r.append_suffix, r.lower, r.capitalize, r.upper
        )
    )

In [21]:
# предскажем правила лемматизации для всех слов, но работать будем только с правилами из трейна

tqdm.pandas(desc="Подготовка правил лемматизации")

for df in (df_train, df_test, df_test_known, df_test_unkn):
    def _foo(row):
        return predict_lemmatize_rule(row["form"], row["lemma"])
    df["lemmatization_rule"] = df.progress_apply(
        _foo,
        axis=1
    )

Подготовка правил лемматизации: 100%|██████████| 20401/20401 [00:00<00:00, 37910.81it/s]


In [22]:
unique_rules = set(df_train["lemmatization_rule"].unique())
len(unique_rules)

4762

In [23]:
id2rule: dict[int, LemmatizeRule]= {
        i: rule for i, rule
        in enumerate(_sort_rules(unique_rules))
    }
rule2id = {rule: i for i, rule in id2rule.items()}

In [32]:
for df in (df_train, df_test, df_test_known, df_test_unkn):
    df["rule_id"] = df["lemmatization_rule"].map(rule2id)

In [ ]:
# почти для всех случаев правила определяются в трейне!
# неопределенные случаи надо заполнить и заценить, что там
df_test_unkn.info()

<class 'pandas.DataFrame'>
Index: 20401 entries, 0 to 255976
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   form                20401 non-null  str    
 1   pos                 20401 non-null  str    
 2   feats               20401 non-null  str    
 3   lemma               20401 non-null  str    
 4   lemmatization_rule  20401 non-null  object 
 5   rule_id             20103 non-null  float64
dtypes: float64(1), object(1), str(4)
memory usage: 1.1+ MB


In [39]:
UNDEFINED_RULE_ID = -100

In [ ]:
df_test_unkn["rule_id"] = df_test_unkn["rule_id"].fillna(UNDEFINED_RULE_ID)

In [40]:
unkn_test_undefined_rule = df_test_unkn[df_test_unkn["rule_id"] == UNDEFINED_RULE_ID]

In [41]:
unkn_test_undefined_rule.shape[0]

298

In [42]:
unkn_test_undefined_rule.head(15)

,form,pos,feats,lemma,lemmatization_rule,rule_id
144,христа,PROPN,Animacy:Anim Case:Gen Gender:Masc NameType:Prs...,Христос,"LemmatizeRule(cut_prefix=0, cut_suffix=1, appe...",-100.0
282,воздушно-,ADJ,Case:Ins Degree:Pos Gender:Masc Number:Sing,воздушно-капельный,"LemmatizeRule(cut_prefix=0, cut_suffix=0, appe...",-100.0
283,капельным,X,No Feats,_,"LemmatizeRule(cut_prefix=9, cut_suffix=0, appe...",-100.0
1437,2,ANUM,NumForm:Combi NumType:Ord,2-й,"LemmatizeRule(cut_prefix=0, cut_suffix=0, appe...",-100.0
2081,Л'айк,NOUN,Animacy:Inan Case:Acc Gender:Masc Number:Sing,лайк,"LemmatizeRule(cut_prefix=0, cut_suffix=4, appe...",-100.0
2106,Подпишись,VERB,Aspect:Perf Mood:Imp Number:Sing Person:2 Tran...,подпиcаться,"LemmatizeRule(cut_prefix=0, cut_suffix=4, appe...",-100.0
2356,ЛА-ЙК,NOUN,Animacy:Inan Case:Nom Gender:Masc Number:Sing,лайк,"LemmatizeRule(cut_prefix=0, cut_suffix=3, appe...",-100.0
3412,не,ADV,Degree:Pos,ненадолго,"LemmatizeRule(cut_prefix=0, cut_suffix=0, appe...",-100.0
4511,мильен,NOUN,Animacy:Inan Case:Nom Gender:Masc Number:Sing,миллион,"LemmatizeRule(cut_prefix=0, cut_suffix=3, appe...",-100.0
6407,судорге,NOUN,Animacy:Inan Case:Loc Gender:Fem Number:Sing,судорога,"LemmatizeRule(cut_prefix=0, cut_suffix=2, appe...",-100.0


In [ ]:
# доля случаев с неопределенным парвилом (правило не встретилось в тренировочных данных)
unkn_test_undefined_rule.shape[0] / df_test_unkn.shape[0]

0.014607127101612665

In [46]:
## top k most frequent rules in train

k = 10
topk_rule_ids = df_train['rule_id'].value_counts().head(k).rename_axis('rule_id').reset_index(name='count')

for row in topk_rule_ids.itertuples(index=False):
    print(f"\n{'='*80}")
    print(f"rule_id: {row.rule_id}  |  count: {row.count}")
    print(f"rule: {id2rule[row.rule_id]}")
    subset = df_train[df_train['rule_id'] == row.rule_id]
    if set(subset["pos"]) == set(df_train["pos"]):
        pos_set = "ALL TAGS!!!"
    else:
        pos_set = sorted(subset['pos'].dropna().unique())
    print(f"pos tags: {pos_set}")
    print('-'*80)
    ex = subset.sample(
        n=min(5, row.count), random_state=42
    )
    for _, r in ex.iterrows():
        print(f"  form={r['form']!r:15}  lemma={r['lemma']!r:15}  pos={r['pos']!r:10}  feats={r['feats']}")


rule_id: 0  |  count: 48886
rule: LemmatizeRule(cut_prefix=0, cut_suffix=0, append_suffix='', lower=False, capitalize=False, upper=False)
pos tags: ['ADJ', 'ADP', 'ADV', 'ADVPRO', 'ANUM', 'AUX', 'CCONJ', 'COM', 'DET', 'INTJ', 'NOUN', 'NUM', 'PARENTH', 'PART', 'PRED', 'PREDPRO', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']
--------------------------------------------------------------------------------
  form='Алма'           lemma='Алма'           pos='PROPN'     feats=Animacy:Anim Case:Nom Gender:Fem NameType:Giv Number:Sing
  form='могущество'     lemma='могущество'     pos='NOUN'      feats=Animacy:Inan Case:Nom Gender:Neut Number:Sing
  form='подгору'        lemma='подгору'        pos='X'         feats=Foreign:Yes
  form='впрямую'        lemma='впрямую'        pos='ADV'       feats=Degree:Pos
  form='пространный'    lemma='пространный'    pos='ADJ'       feats=Case:Nom Degree:Pos Gender:Masc Number:Sing

rule_id: 131  |  count: 19752
rule: LemmatizeRule(cut_prefix=0, cut